# Stats of the dataset

In [2]:
%cd /Users/chihirot/ikema_asr/data

/Users/chihirot/ikema_asr/data


In [ ]:
import datasets
import os
from dotenv import load_dotenv

load_dotenv()

# main_dataset = datasets.load_from_disk("ikema_youtube_asr")
main_dataset = datasets.load_dataset("ctaguchi/ikema_youtube_asr_full",
                                     token=os.getenv("HF_TOKEN"))["train"]
main_dataset

Dataset({
    features: ['audio', 'transcription', 'romaji', 'phoneme', 'start', 'end', 'title', 'id'],
    num_rows: 9842
})

In [72]:
eval_dataset = datasets.load_from_disk("ikema_youtube_asr_test")
eval_dataset

Dataset({
    features: ['audio', 'transcription', 'romaji', 'phoneme', 'start', 'end', 'title'],
    num_rows: 285
})

In [73]:
set(eval_dataset["title"])

{'jugon', 'mimamuibusu'}

In [64]:
titles = set(main_dataset["title"])
len(titles)

94

In [ ]:
lecture_data = main_dataset.

In [74]:
combined_dataset = datasets.concatenate_datasets([main_dataset, eval_dataset])
combined_dataset

Dataset({
    features: ['audio', 'transcription', 'romaji', 'phoneme', 'start', 'end', 'title', 'id'],
    num_rows: 10127
})

In [75]:
from tqdm import tqdm

stats = {}

for sample in tqdm(combined_dataset):
    title = sample["title"]
    if title not in stats.keys():
        stats[title] = {}
    duration = (sample["end"] - sample["start"]) / 1000  # in seconds
    
    if "duration" not in stats[title]:
        stats[title]["duration"] = 0    
    stats[title]["duration"] += duration
    
    if "num_samples" not in stats[title]:
        stats[title]["num_samples"] = 0
    stats[title]["num_samples"] += 1

    if "num_kana" not in stats[title]:
        stats[title]["num_kana"] = 0
    stats[title]["num_kana"] += len(sample["transcription"])

    if "num_romaji" not in stats[title]:
        stats[title]["num_romaji"] = 0
    stats[title]["num_romaji"] += len(sample["romaji"])
    
    if "num_words" not in stats[title]:
        stats[title]["num_words"] = 0
    stats[title]["num_words"] += len(sample["transcription"].split())

stats

100%|██████████| 10127/10127 [01:49<00:00, 92.42it/s]


{'ishiusu': {'duration': 114.51,
  'num_samples': 77,
  'num_kana': 942,
  'num_romaji': 1334,
  'num_words': 203},
 'aisatsu_jikoshoukai1': {'duration': 13.97,
  'num_samples': 11,
  'num_kana': 81,
  'num_romaji': 119,
  'num_words': 19},
 'satatinpura': {'duration': 44.300000000000004,
  'num_samples': 29,
  'num_kana': 382,
  'num_romaji': 517,
  'num_words': 77},
 'dakyau': {'duration': 19.399999999999995,
  'num_samples': 15,
  'num_kana': 127,
  'num_romaji': 202,
  'num_words': 27},
 'haka': {'duration': 74.14,
  'num_samples': 51,
  'num_kana': 618,
  'num_romaji': 886,
  'num_words': 136},
 'suzumebachi': {'duration': 321.325,
  'num_samples': 217,
  'num_kana': 3018,
  'num_romaji': 4338,
  'num_words': 586},
 'takanna': {'duration': 44.99,
  'num_samples': 34,
  'num_kana': 412,
  'num_romaji': 591,
  'num_words': 83},
 'zyaagama': {'duration': 216.23999999999992,
  'num_samples': 143,
  'num_kana': 1877,
  'num_romaji': 2671,
  'num_words': 355},
 'aisatsu_yurinokai': {'du

### Total duration

In [78]:
# total duration
total_duration = sum([stats[title]["duration"] for title in stats.keys()])
total_duration / 3600

4.215115277777778

In [79]:
total_duration

15174.414999999999

## Dictionary dataset

In [ ]:
dict_dataset = datasets.load_dataset("ctaguchi/ikema_dict_asr")["train"]
dict_dataset

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Generating train split: 100%|██████████| 5680/5680 [00:01<00:00, 3068.76 examples/s]


DatasetDict({
    train: Dataset({
        features: ['word_id', 'word', 'audio', 'part_of_speech', 'description', 'example_sentence', 'romaji', 'phoneme'],
        num_rows: 5680
    })
})

In [87]:
from tqdm import tqdm

stats = {"duration": 0, "num_samples": 0, "num_kana": 0, "num_romaji": 0, "num_words": 0}

for sample in tqdm(dict_dataset):
    sr = sample["audio"]["sampling_rate"]
    duration = sample["audio"]["array"].shape[0] / sr  # in seconds
    stats["duration"] += duration
    
    stats["num_samples"] += 1

    stats["num_kana"] += len(sample["word"])
    
    stats["num_romaji"] += len(sample["romaji"])
    
    stats["num_words"] += len(sample["word"].split())

stats

100%|██████████| 5680/5680 [00:40<00:00, 141.65it/s]


{'duration': 5378.762625000046,
 'num_samples': 5680,
 'num_kana': 23414,
 'num_romaji': 36243,
 'num_words': 5928}

In [88]:
stats["duration"] / 3600

1.4941007291666795

### Inspection

In [6]:
sorted(list(stats.keys()))

['aisatsu_jikoshoukai1',
 'aisatsu_jikoshoukai2',
 'aisatsu_jikoshoukai_hyoujun',
 'aisatsu_otoori',
 'aisatsu_roujinkai',
 'aisatsu_yurinokai',
 'aman',
 'avvansu',
 'barazyan',
 'bippii',
 'butaniku_yasai_nimono',
 'buugii',
 'dakyau',
 'fukyagi',
 'gazugii',
 'harimizuutaki',
 'ikemaoohashi',
 'ishiusu',
 'kaichou_shokureki_hiroyuki',
 'kaichou_shokureki_tadashi',
 'kami',
 'kkutsugii',
 'kuwazuimo',
 'kyuukouminkan',
 'kyuukouminkan_ura',
 'mamisuimai',
 'mingu_andira',
 'mingu_huta',
 'miyakosoba',
 'mutsuusa',
 'nakamautaki',
 'nakasunituimyaa',
 'nevsky',
 'ngi',
 'niguu',
 'nimono',
 'saguna',
 'satatinpura',
 'shiisaa',
 'susuki',
 'suuni',
 'suzumebachi',
 'taka',
 'takanna',
 'ugan',
 'usu',
 'waanimun',
 'yasaiitame',
 'zyaagama',
 'zzakugii']

In [50]:
stats["usu"]

{'duration': 111.03000000000003,
 'num_samples': 69,
 'num_kana': 1105,
 'num_romaji': 1590,
 'num_words': 202}

In [ ]:
title_map = {
    # I0045_若いころの話 is unannotated, but this is identical to tamunu
    "koebitori": "I0045_koebitori", # I0045_小エビ取り is unannotated -> done
    "yaafutsu": "I0045_yaafucI", # this is not yet included in the dataset -> done
    "namakonohanashi": "I0045_keito_namako", # I0045_keito_namako is unannotated -> done
    "utakinohanashi": "I0096_nimaigai_utaki", # this is not yet included in the dataset -> done
    "tamunu": "I0045_tamunu", # this is not yet included in the dataset -> done
    "nimaigai_senso": "I0096_nimaigai_senso", # I0096_nimaigai2_senso is unannotated -> done
    "sinatui": "I0096_sinatui", # this is not yet included in the dataset -> done
    "nazuke": "I0097_kaicho_naa", # this is not yet included in the dataset -> done
    "aisatsu": "I0139_aisacI", # this is not yet included in the dataset -> done
    "masImuiuya": "I0307_masImuiuya_Va", # I0307_masImuiuya_Va is unannotated -> done
    "souchou_imo": "I0391_ssakai2", # this is not yet included in the dataset -> done
    "kusakari": "I0391_ssakai", # this is not yet included in the dataset -> done
    "hiroyuki_digital_museum": "I0396_museumaisatu_kocho", # this is not yet included in the dataset -> done
    "ooura_bay": "I0398_ooura", # this is not yet included in the dataset -> done
    "zyanmiga": "I0399_janmiga", # this is not yet included in the dataset -> done
    "hidagaa": "I0400_hidagaa", # this is not yet included in the dataset -> done
    "bubakariishi": "I0401_nintozeiseki", # this is not yet included in the dataset -> done
    "tadashi_digital_museum": "I0402_kaichoaisatsu", # this is not yet included in the dataset -> done
    "mitsuyoshi_digital_museum": "I0403_azachoaisatsu", # this is not yet included in the dataset -> done
    "bituriiduu": "I0404_bituriiduu_Vl", # I0404_bituriiduu_Vl is unannotated -> done
    "yamatugaa": "I0407_yamatugaa", # this is not yet included in the dataset -> done
    "bissiutaki": "I0409b_bissi_utaki", # this is not yet included in the dataset -> done
    "irabu_kankou3": "I0410_irabukankouVd", # this is not yet included in the dataset -> done
    "irabu_kankou2": "I0410_irabukankouVc", # this is not yet included in the dataset -> done
    "irabu_kankou1": "I0410_irabukankouVa", # this is not yet included in the dataset -> done
    "mamegahana": "I0413_mamigahana_honban", # this is not yet included in the dataset -> done
    "toujintorainouta": "I0415_toojintorai", # this is not yet included in the dataset -> done
    "wakimizu": "I0482_pic_wakimizu", # I0482_pic_wakimizu is unannotated -> done
    "usu": "I0482_usI",
    "ngi": "I0482_ngi",
    "mutsuusa": "I0482_pic_mucIusa",
    "mancyuu": "I0482_manchu", # this is not yet included in the dataset -> done
    "masugita": "I0482_masIgita", # this is not yet included in the dataset -> done
    "maaninuhigi": "I0482_maaninuhigi", # this is not yet included in the dataset -> done
    "maani": "I0482_maani", # this is not yet included in the dataset -> done
    "ffuzyatatinpura": "I0482_fuzyatatinpura", # is unannotated -> done
    "buuzu": "I0482_buuzI", # this is not yet included in the dataset -> done
    "buugii": "I0482_pic_buugii",
    "kuwazuimo": "I0482_byuuigassa",
    "bippii": "I0482_bippii",
    "bikiyadumura": "I0482_bikiyadumura", # this is not yet included in the dataset -> done
    "pii": "I0482_pii", # this is not yet included in the dataset -> done
    "bantsugii": "I0482_bancIgii", # this is not yet included in the dataset -> done
    "barazyan": "I0482_pic_barazan",
    "basa": "I0482_basa", # this is not yet included in the dataset -> done
    "haka": "I0482_haka", # this is not yet included in the dataset -> done
    "baaki": "I482_baaki", # this is not yet included in the dataset -> done
    "niguu": "I0482_niguu",
    "takanna": "I0482_pic_takanna",
    "taka": "I0482_pic_taka",
    "suuni": "I0412_suuni",
    "satatinpura": "I0483_0414_satatinpura",
    "zzakugii": "I0482_pic_zzakugii",
    "kubazuu": "I0482_kubazII", # this is not yet included in the dataset -> done
    "susuki": "I0482_gisIcI",
    "kami": "I0482_kami",
    "gazugii": "I0482_gazIhanagii",
    "ugan": "I0482_ugan",
    "oogomadara": "I0482_ayabasa", # this is not yet included in the dataset -> done
    "adanbasaba": "I0482_adanbasaba", # this is not yet included in the dataset -> done
    "yasaiitame": "I0483_0277_yasaiitame",
    "waatuhunya": "I483_waatuhunya", # waatuhunya is lacking -> done
    "butaniku_yasai_nimono": "I0483_0276_waa",
    "ishiusu": "I0483_0405_isIusI",
    "miyakosoba": "I0483_0527_suba",
    "waanimun": "I0483_0412_waanimun",
    "mamisuimai": "I0483_0688_mamisuimai",
    "fukyagi": "I0483_0409_fukyagi",
    "dakyau": "I0483_dakyau",
    "suzumebachi": "I0482_pc_taummabasI",
    "juusamai": "I0483_0528_zyuusamai", # this is not yet included in the dataset -> done
    "saguna": "I0483_0678_saguna",
    "kkutsugii": "I0482_kkucIgii",
    "kayayaa": "I0482_kayayaa",
    "avvansu": "I0483_0222_avvansu",
    "utaki": "I0483_utaki", # I0483_うたき is unannotated -> done
    "cIccyu": "I0487_cIccyu", # cIccyu is lacking -> done
    "shiisaa": "I0488_shiisaa",
    "guunya": "I0486_guunya", # guunya are lacking -> done
    "sabaucIgaa": "I0490_sabaucIgaa", # sabaucIgaa is lacking -> done
    "aman": "I0490_aman",
    "ikemaoohashi": "I0490_ikemaoohashi",
    "kaichou_shokureki_hiroyuki": "I0496_kaichosyokureki_hiroyuki",
    "kaichou_shokureki_tadashi": "I0496_kaichosyokureki_tadashi",
    "nakasunituimyaa": "I0501_nakasonetuimyaa",
    "harimizuutaki": "I0502_harimizuutaki",
    "nevsky": "I0503_Nevskynohi",
    "aisatsu_jikoshoukai1": "I0506_aisatsutehon_Va",
    "aisatsu_jikoshoukai2": "I0506_aisatsutehon_Vb",
    "aisatsu_jikoshoukai_hyoujun": "I0506_aisatsutehon_Vc",
    "aisatsu_otoori": "I0506_aisatsutehon_Vd",
    "aisatsu_roujinkai": "I0506_aisatsutehon_Ve",
    "aisatsu_yurinokai": "I0506_aisatsutehon_Vf",
    "mingu_huta": "I0507_mingukaisetsu_Va",
    "mingu_andira": "I0507_mingukaisetsu_Vb",
    "nakamautaki": "I0508_nakamautaki_V",
    "zyaagama": "I0509_zyaagama_V",
    "kyuukouminkan": "I0510_kyuukoominkan_Va",
    "kyuukouminkan_ura": "I0510_kyuukoominkan_Vb",
}

SyntaxError: invalid syntax (2762349506.py, line 31)

In [ ]:
dict_dataset["train"][-1]

{'word_id': 5648,
 'word': 'んﾟんびつ',
 'audio': {'path': 'ikmvoc_5648.wav',
  'array': array([-1.49636097e-04, -7.64025894e-04, -6.59112214e-04, ...,
          6.84034595e-05, -2.80044900e-04,  0.00000000e+00]),
  'sampling_rate': 16000},
 'part_of_speech': '動詞',
 'description': '踏んづける',
 'example_sentence': 'いんぬ\u3000っそぅー\u3000んﾟんびきーにゃーん（犬の糞を踏んづけてしまった）。',
 'romaji': 'N̥NbicI',
 'phoneme': 'm̥mbitsɨ'}